# Preservation of Movies That Matter Festival Archive

Authors: Luiza Filip, Minja Cvetkovski, Andreea Ion 

## Aim of this notebook

In this notebook, we aim to preserve all 1202 movies screened at the MtMF from 2009-2026. Therefore, here we utilize different webscraping tools in order to create our <span style='color:lightblue'>csv</span> file.

### Importing libraries
First and foremost, in the cell below, we will import all the libraries needed for this project.

In [34]:
import pandas as pd
import requests
import time
from bs4 import BeautifulSoup
import re 
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options 
from selenium.webdriver.support.ui import WebDriverWait 
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service

Now that all the libraries have been imported, it is time to create our dataframe.
But before we do that, we need to ensure of a few things:

## Setting Up 
We will first use a variable called __base_link__ which will keep our scraper consistent (same language path, same route, same hash-based state) and makes it easy to change the target later without hurting through the code.

In [35]:
base_link = "https://moviesthatmatter.nl/en/festival/archive/film-archive/#/"

Now that we got our base set up, let's run a function that starts our Chrome WebDriver and sets a consistent window size for stable scraping, so we can begin interacting with the website. :)

In [36]:
#launching and returning a configured chrome webdriver with a page-load timeout
def start_driver(headless=True):
    chrome_options = Options()
    if headless:
        chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--window-size=1400,900")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    #selenium manager auto-handles the driver
    driver = webdriver.Chrome(options=chrome_options)
    driver.set_page_load_timeout(60)
    return driver

Once the website is launched, a cookies tab pops up which normally should be manually clicked in order to begin scraping. Therefore, we decided to implement a function that automatically accepts the cookies button, so we do not have to worry about it and ensure smooth execution.

In [37]:
def accept_cookies(driver, timeout=10):
    WebDriverWait(driver, timeout).until(
        #we already know how the cookie button looks like so we can write it like this:
        EC.element_to_be_clickable((By.XPATH, "//button[normalize-space()='Accept all cookies']"))
    ).click()

## Scraping URLs
Now that we've got all of that out of our way, we need to start scraping all the movie URLs by looping through every page in the film archive, waiting for the results to load, extracting each link, and the de-duplicating them into one clean list (total of 1202).

In [38]:
def wait_for_results(driver):
    #strong wait doesnt continue until at least one link exists
    WebDriverWait(driver, 40).until(
        lambda d: len(d.find_elements(By.CSS_SELECTOR, "a[href*='/en/film/']")) > 0
    )

def get_last_page_number(driver):
    #pagination "last" button has href like "#/121"
    last = WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "a.pagination-item.pagination-last"))
    )
    href = last.get_attribute("href") or ""
    m = re.search(r"#/(\d+)", href)
    if not m:
        raise RuntimeError(f"Could not parse last page number from: {href}")
    return int(m.group(1))

def extract_movie_links_from_current_page(driver):
    anchors = driver.find_elements(By.CSS_SELECTOR, "a[href*='/en/film/']")
    links = []
    for a in anchors:
        href = a.get_attribute("href")
        if href:
            links.append(href.split("#")[0].rstrip("/"))
    #de-duplicate while preserving order
    return list(dict.fromkeys(links))



def scrape_all_movie_urls(headless=True, polite_sleep=0.2):
    driver = start_driver(headless=headless)
    all_links = []

    try:
        #load page 1 to detect last page
        driver.get(base_link + "1")
        accept_cookies(driver)
        wait_for_results(driver)

        last_page = get_last_page_number(driver)
        print(f"Detected last page: {last_page}")

        for page in range(1, last_page + 1):
            driver.get(f"{base_link}{page}")
            wait_for_results(driver)

            links = extract_movie_links_from_current_page(driver)
            all_links.extend(links)

            print(f"Currently on page {page}/{last_page}: {len(all_links)} total collected URLs.")
            time.sleep(polite_sleep)


    finally:
        driver.quit()

    #final de-dup across pages
    all_links = list(dict.fromkeys(all_links)) 
    return pd.DataFrame({"movie_url": all_links})

Now that our scraping functions are ready, we can run the URLs, verifying how many movie links we collected (if all goes well it should be 1202).

In [39]:
df_movies = scrape_all_movie_urls(headless=True)
print("Total rows:", len(df_movies))   
df_movies.head()

Detected last page: 121
Currently on page 1/121: 10 total collected URLs.
Currently on page 2/121: 20 total collected URLs.
Currently on page 3/121: 30 total collected URLs.
Currently on page 4/121: 40 total collected URLs.
Currently on page 5/121: 50 total collected URLs.
Currently on page 6/121: 60 total collected URLs.
Currently on page 7/121: 70 total collected URLs.
Currently on page 8/121: 80 total collected URLs.
Currently on page 9/121: 90 total collected URLs.
Currently on page 10/121: 100 total collected URLs.
Currently on page 11/121: 110 total collected URLs.
Currently on page 12/121: 120 total collected URLs.
Currently on page 13/121: 130 total collected URLs.
Currently on page 14/121: 140 total collected URLs.
Currently on page 15/121: 150 total collected URLs.
Currently on page 16/121: 160 total collected URLs.
Currently on page 17/121: 170 total collected URLs.
Currently on page 18/121: 180 total collected URLs.
Currently on page 19/121: 190 total collected URLs.
Curren

,movie_url
0,https://moviesthatmatter.nl/en/film/chicagogir...
1,https://moviesthatmatter.nl/en/film/10-conditi...
2,https://moviesthatmatter.nl/en/film/107-mothers
3,https://moviesthatmatter.nl/en/film/12-years-a...
4,https://moviesthatmatter.nl/en/film/143-sahara...


Looks like we collected a total of 1202 rows, matching the number of movies in the MtMF archive. Cool!

## Scraping Titles 
Now we are going to extract the titles from each URLs and adding a new column to our DataFrame. 

In [40]:
#make an empty list to store titles
titles = []

#loop through each URL
for url in df_movies["movie_url"]:
    page = requests.get(url)
    soup = BeautifulSoup(page.text, "html.parser")

    #find the title (h1)
    h1 = soup.find("h1")

    #appending the title
    if h1:
        titles.append(h1.get_text(strip=True))
    else:
        titles.append(pd.NA)

#add list as new column
df_movies["title"] = titles

df_movies.head(6)


,movie_url,title
0,https://moviesthatmatter.nl/en/film/chicagogir...,#chicagoGirl – The Social Network Takes on a D...
1,https://moviesthatmatter.nl/en/film/10-conditi...,The 10 Conditions of Love
2,https://moviesthatmatter.nl/en/film/107-mothers,107 Mothers
3,https://moviesthatmatter.nl/en/film/12-years-a...,12 Years A Slave
4,https://moviesthatmatter.nl/en/film/143-sahara...,143 Sahara Street
5,https://moviesthatmatter.nl/en/film/15-minutes...,15 minutes – The Dvor Massacre


Alright, now we just collected all the titles. Let's move on to our next objectives in the preservation process.

## Scraping Descriptions
Now we are going to scrape the description of each movie:

In [41]:
#make an empty list to store descriptions
descriptions = []

#loop through each URL
for url in df_movies["movie_url"]:
    page = requests.get(url)
    soup = BeautifulSoup(page.text, "html.parser")

    #find the description div 
    desc = soup.find("div", class_="film-detail__the-content")

    #appending the description
    if desc:
        descriptions.append(desc.text.strip())
    #If a movie does not have one, than we add NA
    else:
        descriptions.append(pd.NA)

#add list as new column
df_movies["description"] = descriptions

df_movies.head(6)

,movie_url,title,description
0,https://moviesthatmatter.nl/en/film/chicagogir...,#chicagoGirl – The Social Network Takes on a D...,The 19-year-old Ala’a Basatneh helps coordinat...
1,https://moviesthatmatter.nl/en/film/10-conditi...,The 10 Conditions of Love,Rebiya Kadeer once embodied China’s economic s...
2,https://moviesthatmatter.nl/en/film/107-mothers,107 Mothers,"In a Ukrainian women’s prison, inmate Lesya ha..."
3,https://moviesthatmatter.nl/en/film/12-years-a...,12 Years A Slave,Touching and thrilling movie based on the memo...
4,https://moviesthatmatter.nl/en/film/143-sahara...,143 Sahara Street,"In her small shop, in the vast Algerian Sahara..."
5,https://moviesthatmatter.nl/en/film/15-minutes...,15 minutes – The Dvor Massacre,"In the summer of 1995\nJørgen Kold, Danish off..."


Hmm, looks like in row 5 we can spot a __\n__. It's best to clean the columns now:

In [42]:
df_movies["description"] = (
    df_movies["description"]
      .astype("string")
      .str.replace(r"\s+", " ", regex=True)   #turning \n, \t, multiple spaces into single space
      .str.strip())
df_movies.head(10)

,movie_url,title,description
0,https://moviesthatmatter.nl/en/film/chicagogir...,#chicagoGirl – The Social Network Takes on a D...,The 19-year-old Ala’a Basatneh helps coordinat...
1,https://moviesthatmatter.nl/en/film/10-conditi...,The 10 Conditions of Love,Rebiya Kadeer once embodied China’s economic s...
2,https://moviesthatmatter.nl/en/film/107-mothers,107 Mothers,"In a Ukrainian women’s prison, inmate Lesya ha..."
3,https://moviesthatmatter.nl/en/film/12-years-a...,12 Years A Slave,Touching and thrilling movie based on the memo...
4,https://moviesthatmatter.nl/en/film/143-sahara...,143 Sahara Street,"In her small shop, in the vast Algerian Sahara..."
5,https://moviesthatmatter.nl/en/film/15-minutes...,15 minutes – The Dvor Massacre,"In the summer of 1995 Jørgen Kold, Danish offi..."
6,https://moviesthatmatter.nl/en/film/17-blocks,17 Blocks,Two decades in the life of the Sanford-Durant ...
7,https://moviesthatmatter.nl/en/film/3-1-2-minu...,"3 1/2 Minutes, Ten Bullets",Electrifying documentary thriller about Afro-A...
8,https://moviesthatmatter.nl/en/film/33-days,33 Days,"In the 2006 war between Israel and Lebanon, ar..."
9,https://moviesthatmatter.nl/en/film/5-broken-c...,5 Broken Cameras,The major winner at IDFA 2011. Personal testim...


## Scraping Themes
We will scrape all the themes appearing under each movie, making sure to also keep them as a list of strings since some movies have more than one theme.

In [43]:
#empty list to store themes
themes = []

#looping through each URL
for url in df_movies["movie_url"]:
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    theme = pd.NA  
    #finding themes div
    info_list = soup.find("div", class_="education-info__list")
    if info_list:
        labels = info_list.find_all("div", class_="education-info__item-name")
        values = info_list.find_all("div", class_="education-info__item-values")

        for label, value in zip(labels, values):
            if label.get_text(strip=True).lower() == "themes":
                found = [
                    span.get_text(strip=True)
                    for span in value.find_all("span", class_="education-info__item-value")
                ]
                #adding NA if it doesn't exist
                theme = found if found else pd.NA  
                break
    #appending the found theme to the list
    themes.append(theme)

#adding a new column to the dataframe
df_movies["themes"] = themes
df_movies.head()

,movie_url,title,description,themes
0,https://moviesthatmatter.nl/en/film/chicagogir...,#chicagoGirl – The Social Network Takes on a D...,The 19-year-old Ala’a Basatneh helps coordinat...,"[Activism and human rights defenders, Democrac..."
1,https://moviesthatmatter.nl/en/film/10-conditi...,The 10 Conditions of Love,Rebiya Kadeer once embodied China’s economic s...,"[Activism and human rights defenders, Culture ..."
2,https://moviesthatmatter.nl/en/film/107-mothers,107 Mothers,"In a Ukrainian women’s prison, inmate Lesya ha...","[Prison and detention, Women's rights]"
3,https://moviesthatmatter.nl/en/film/12-years-a...,12 Years A Slave,Touching and thrilling movie based on the memo...,"[Democracy and politics, Racism and/or discrim..."
4,https://moviesthatmatter.nl/en/film/143-sahara...,143 Sahara Street,"In her small shop, in the vast Algerian Sahara...",[Women's rights]


## Scraping Relevant Countries
Now we are going to scrape the section named Relevant countries, and just like in the case of themes, multiple countries can occur, so we treat them as a list of strings as well:

In [44]:
#empty list to store the relevant countries
relevant_countries = []

#looping through each URl
for url in df_movies["movie_url"]:
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    countries = pd.NA  
    
    info_list = soup.find("div", class_="education-info__list")
    if info_list:
        labels = info_list.find_all("div", class_="education-info__item-name")
        values = info_list.find_all("div", class_="education-info__item-values")

        for label, value in zip(labels, values):
            if label.get_text(strip=True).lower() == "relevant countries":
                found = [
                    span.get_text(strip=True)
                    for span in value.find_all("span", class_="education-info__item-value")
                ]
                countries = found if found else pd.NA  
                break

    relevant_countries.append(countries)
#adding new column to the dataframe
df_movies["relevant_countries"] = relevant_countries


## Scraping MtMF 
This refers to the year in which the movies were screened at the festival. We will keep using the same format as before for scraping and adding column to the dataframe:

In [45]:
#empty list to store MtMF year
mtmf_years = []

#looping though each URL
for url in df_movies["movie_url"]:
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    mtmf_year = pd.NA

    info_list = soup.find("div", class_="education-info__list")
    if info_list:
        labels = info_list.find_all("div", class_="education-info__item-name")
        values = info_list.find_all("div", class_="education-info__item-values")

        for label, value in zip(labels, values):
            if label.get_text(strip=True).lower() == "mtmf":
                span = value.find("span", class_="education-info__item-value")
                if span:
                    txt = span.get_text(strip=True)
                    #convert to numeric
                    mtmf_year = int(txt) if txt.isdigit() else txt
                break
    #appending the year to the list
    mtmf_years.append(mtmf_year)

df_movies["mtmf_year"] = mtmf_years
df_movies.head()

,movie_url,title,description,themes,relevant_countries,mtmf_year
0,https://moviesthatmatter.nl/en/film/chicagogir...,#chicagoGirl – The Social Network Takes on a D...,The 19-year-old Ala’a Basatneh helps coordinat...,"[Activism and human rights defenders, Democrac...","[Syria, United States]",2014
1,https://moviesthatmatter.nl/en/film/10-conditi...,The 10 Conditions of Love,Rebiya Kadeer once embodied China’s economic s...,"[Activism and human rights defenders, Culture ...","[China, Taiwan, Tibet, Turkey]",2010
2,https://moviesthatmatter.nl/en/film/107-mothers,107 Mothers,"In a Ukrainian women’s prison, inmate Lesya ha...","[Prison and detention, Women's rights]",[Ukraine],2022
3,https://moviesthatmatter.nl/en/film/12-years-a...,12 Years A Slave,Touching and thrilling movie based on the memo...,"[Democracy and politics, Racism and/or discrim...",[United States],2014
4,https://moviesthatmatter.nl/en/film/143-sahara...,143 Sahara Street,"In her small shop, in the vast Algerian Sahara...",[Women's rights],[Algeria],2020


Now that we have our main 3: themes, relevand countries and MtMF year, let's scrape all of the information from the __Credits__ section.

## Scraping Director

In [46]:
#empty list to store directors of movies
directors = []

for url in df_movies["movie_url"]:
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    director = pd.NA

    credits = soup.find("dl", class_="data-list--details")
    if credits:
        items = credits.find_all("dt")
        values = credits.find_all("dd")

        for item, value in zip(items, values):
            if item.get_text(strip=True).lower() == "director":
                # grab all text pieces inside the <dd>
                found = [
                    s.strip()
                    for s in value.stripped_strings
                    if s.strip()
                ]
                director = found if found else pd.NA
                break

    directors.append(director)

df_movies["director"] = directors
df_movies.head(10)

,movie_url,title,description,themes,relevant_countries,mtmf_year,director
0,https://moviesthatmatter.nl/en/film/chicagogir...,#chicagoGirl – The Social Network Takes on a D...,The 19-year-old Ala’a Basatneh helps coordinat...,"[Activism and human rights defenders, Democrac...","[Syria, United States]",2014,[Joe Piscatella]
1,https://moviesthatmatter.nl/en/film/10-conditi...,The 10 Conditions of Love,Rebiya Kadeer once embodied China’s economic s...,"[Activism and human rights defenders, Culture ...","[China, Taiwan, Tibet, Turkey]",2010,[Jeff Daniels]
2,https://moviesthatmatter.nl/en/film/107-mothers,107 Mothers,"In a Ukrainian women’s prison, inmate Lesya ha...","[Prison and detention, Women's rights]",[Ukraine],2022,[Péter Kerekes]
3,https://moviesthatmatter.nl/en/film/12-years-a...,12 Years A Slave,Touching and thrilling movie based on the memo...,"[Democracy and politics, Racism and/or discrim...",[United States],2014,[Steve McQueen]
4,https://moviesthatmatter.nl/en/film/143-sahara...,143 Sahara Street,"In her small shop, in the vast Algerian Sahara...",[Women's rights],[Algeria],2020,[Hassen Ferhani]
5,https://moviesthatmatter.nl/en/film/15-minutes...,15 minutes – The Dvor Massacre,"In the summer of 1995 Jørgen Kold, Danish offi...",[War and armed conflict],"[Bosnia and Herzegovina, Croatia, Denmark]",2016,"[Georg Larsen, Kasper Vedsmand]"
6,https://moviesthatmatter.nl/en/film/17-blocks,17 Blocks,Two decades in the life of the Sanford-Durant ...,"[Crime, Drugs and drug trafficking, Youth and ...",[United States],2020,[Davy Rothbart]
7,https://moviesthatmatter.nl/en/film/3-1-2-minu...,"3 1/2 Minutes, Ten Bullets",Electrifying documentary thriller about Afro-A...,"[Justice and law, Racism and/or discrimination...",[United States],2016,[Marc Silver]
8,https://moviesthatmatter.nl/en/film/33-days,33 Days,"In the 2006 war between Israel and Lebanon, ar...","[Crimes against humanity and war crimes, Migra...",[Lebanon],2009,[Mai Masri]
9,https://moviesthatmatter.nl/en/film/5-broken-c...,5 Broken Cameras,The major winner at IDFA 2011. Personal testim...,"[Activism and human rights defenders, House & ...","[Israel, Palestine]",2012,"[Emad Burnat, Guy Davidi]"


## Scraping Producer

In [47]:
producers = []

for url in df_movies["movie_url"]:
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    producer = pd.NA

    credits = soup.find("dl", class_="data-list--details")
    if credits:
        items = credits.find_all("dt")
        values = credits.find_all("dd")

        for item, value in zip(items, values):
            if item.get_text(strip=True).lower() == "producer":
                found = [s.strip() for s in value.stripped_strings if s.strip()]
                producer = found if found else pd.NA
                break

    producers.append(producer)

df_movies["producer"] = producers
df_movies.head(10)


,movie_url,title,description,themes,relevant_countries,mtmf_year,director,producer
0,https://moviesthatmatter.nl/en/film/chicagogir...,#chicagoGirl – The Social Network Takes on a D...,The 19-year-old Ala’a Basatneh helps coordinat...,"[Activism and human rights defenders, Democrac...","[Syria, United States]",2014,[Joe Piscatella],"[Joe Piscatella, Mike Rinehart]"
1,https://moviesthatmatter.nl/en/film/10-conditi...,The 10 Conditions of Love,Rebiya Kadeer once embodied China’s economic s...,"[Activism and human rights defenders, Culture ...","[China, Taiwan, Tibet, Turkey]",2010,[Jeff Daniels],[John Lewis]
2,https://moviesthatmatter.nl/en/film/107-mothers,107 Mothers,"In a Ukrainian women’s prison, inmate Lesya ha...","[Prison and detention, Women's rights]",[Ukraine],2022,[Péter Kerekes],[Ivan Ostrochovský]
3,https://moviesthatmatter.nl/en/film/12-years-a...,12 Years A Slave,Touching and thrilling movie based on the memo...,"[Democracy and politics, Racism and/or discrim...",[United States],2014,[Steve McQueen],"[Dede Gardner, Anthony Katagas]"
4,https://moviesthatmatter.nl/en/film/143-sahara...,143 Sahara Street,"In her small shop, in the vast Algerian Sahara...",[Women's rights],[Algeria],2020,[Hassen Ferhani],<NA>
5,https://moviesthatmatter.nl/en/film/15-minutes...,15 minutes – The Dvor Massacre,"In the summer of 1995 Jørgen Kold, Danish offi...",[War and armed conflict],"[Bosnia and Herzegovina, Croatia, Denmark]",2016,"[Georg Larsen, Kasper Vedsmand]",[Anne Köhncke]
6,https://moviesthatmatter.nl/en/film/17-blocks,17 Blocks,Two decades in the life of the Sanford-Durant ...,"[Crime, Drugs and drug trafficking, Youth and ...",[United States],2020,[Davy Rothbart],<NA>
7,https://moviesthatmatter.nl/en/film/3-1-2-minu...,"3 1/2 Minutes, Ten Bullets",Electrifying documentary thriller about Afro-A...,"[Justice and law, Racism and/or discrimination...",[United States],2016,[Marc Silver],"[Orlando Bagwell, Su Patel]"
8,https://moviesthatmatter.nl/en/film/33-days,33 Days,"In the 2006 war between Israel and Lebanon, ar...","[Crimes against humanity and war crimes, Migra...",[Lebanon],2009,[Mai Masri],<NA>
9,https://moviesthatmatter.nl/en/film/5-broken-c...,5 Broken Cameras,The major winner at IDFA 2011. Personal testim...,"[Activism and human rights defenders, House & ...","[Israel, Palestine]",2012,"[Emad Burnat, Guy Davidi]",<NA>


## Scraping Year
Now we are going to scrape the years (release) of each movie:

In [48]:
years = []

for url in df_movies["movie_url"]:
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    year = pd.NA
    credits = soup.find("dl", class_="data-list--details")

    if credits:
        items = credits.find_all("dt")
        values = credits.find_all("dd")

        for item, value in zip(items, values):
            if item.get_text(strip=True).lower() == "year":
                txt = value.get_text(strip=True)
                year = txt if txt else pd.NA
                break

    years.append(year)

df_movies["years"] = years  

df_movies.head(10)

,movie_url,title,description,themes,relevant_countries,mtmf_year,director,producer,years
0,https://moviesthatmatter.nl/en/film/chicagogir...,#chicagoGirl – The Social Network Takes on a D...,The 19-year-old Ala’a Basatneh helps coordinat...,"[Activism and human rights defenders, Democrac...","[Syria, United States]",2014,[Joe Piscatella],"[Joe Piscatella, Mike Rinehart]",2013
1,https://moviesthatmatter.nl/en/film/10-conditi...,The 10 Conditions of Love,Rebiya Kadeer once embodied China’s economic s...,"[Activism and human rights defenders, Culture ...","[China, Taiwan, Tibet, Turkey]",2010,[Jeff Daniels],[John Lewis],2009
2,https://moviesthatmatter.nl/en/film/107-mothers,107 Mothers,"In a Ukrainian women’s prison, inmate Lesya ha...","[Prison and detention, Women's rights]",[Ukraine],2022,[Péter Kerekes],[Ivan Ostrochovský],2021
3,https://moviesthatmatter.nl/en/film/12-years-a...,12 Years A Slave,Touching and thrilling movie based on the memo...,"[Democracy and politics, Racism and/or discrim...",[United States],2014,[Steve McQueen],"[Dede Gardner, Anthony Katagas]",2013
4,https://moviesthatmatter.nl/en/film/143-sahara...,143 Sahara Street,"In her small shop, in the vast Algerian Sahara...",[Women's rights],[Algeria],2020,[Hassen Ferhani],<NA>,2019
5,https://moviesthatmatter.nl/en/film/15-minutes...,15 minutes – The Dvor Massacre,"In the summer of 1995 Jørgen Kold, Danish offi...",[War and armed conflict],"[Bosnia and Herzegovina, Croatia, Denmark]",2016,"[Georg Larsen, Kasper Vedsmand]",[Anne Köhncke],2015
6,https://moviesthatmatter.nl/en/film/17-blocks,17 Blocks,Two decades in the life of the Sanford-Durant ...,"[Crime, Drugs and drug trafficking, Youth and ...",[United States],2020,[Davy Rothbart],<NA>,2019
7,https://moviesthatmatter.nl/en/film/3-1-2-minu...,"3 1/2 Minutes, Ten Bullets",Electrifying documentary thriller about Afro-A...,"[Justice and law, Racism and/or discrimination...",[United States],2016,[Marc Silver],"[Orlando Bagwell, Su Patel]",2015
8,https://moviesthatmatter.nl/en/film/33-days,33 Days,"In the 2006 war between Israel and Lebanon, ar...","[Crimes against humanity and war crimes, Migra...",[Lebanon],2009,[Mai Masri],<NA>,2007
9,https://moviesthatmatter.nl/en/film/5-broken-c...,5 Broken Cameras,The major winner at IDFA 2011. Personal testim...,"[Activism and human rights defenders, House & ...","[Israel, Palestine]",2012,"[Emad Burnat, Guy Davidi]",<NA>,2011


Cool cool cool, you got the years of release for each movie
...
as a string 

In [50]:
#extracting a 4-digit year if present, then convert to nullable Int64
df_movies["years"] = (
    df_movies["years"]
      .astype("string")
      .str.extract(r"((?:19|20)\d{2})", expand=False)
)

df_movies["years"] = pd.to_numeric(df_movies["years"], errors="coerce").astype("Int64")


In [52]:
df_movies.head(10)

,movie_url,title,description,themes,relevant_countries,mtmf_year,director,producer,years
0,https://moviesthatmatter.nl/en/film/chicagogir...,#chicagoGirl – The Social Network Takes on a D...,The 19-year-old Ala’a Basatneh helps coordinat...,"[Activism and human rights defenders, Democrac...","[Syria, United States]",2014,[Joe Piscatella],"[Joe Piscatella, Mike Rinehart]",2013
1,https://moviesthatmatter.nl/en/film/10-conditi...,The 10 Conditions of Love,Rebiya Kadeer once embodied China’s economic s...,"[Activism and human rights defenders, Culture ...","[China, Taiwan, Tibet, Turkey]",2010,[Jeff Daniels],[John Lewis],2009
2,https://moviesthatmatter.nl/en/film/107-mothers,107 Mothers,"In a Ukrainian women’s prison, inmate Lesya ha...","[Prison and detention, Women's rights]",[Ukraine],2022,[Péter Kerekes],[Ivan Ostrochovský],2021
3,https://moviesthatmatter.nl/en/film/12-years-a...,12 Years A Slave,Touching and thrilling movie based on the memo...,"[Democracy and politics, Racism and/or discrim...",[United States],2014,[Steve McQueen],"[Dede Gardner, Anthony Katagas]",2013
4,https://moviesthatmatter.nl/en/film/143-sahara...,143 Sahara Street,"In her small shop, in the vast Algerian Sahara...",[Women's rights],[Algeria],2020,[Hassen Ferhani],<NA>,2019
5,https://moviesthatmatter.nl/en/film/15-minutes...,15 minutes – The Dvor Massacre,"In the summer of 1995 Jørgen Kold, Danish offi...",[War and armed conflict],"[Bosnia and Herzegovina, Croatia, Denmark]",2016,"[Georg Larsen, Kasper Vedsmand]",[Anne Köhncke],2015
6,https://moviesthatmatter.nl/en/film/17-blocks,17 Blocks,Two decades in the life of the Sanford-Durant ...,"[Crime, Drugs and drug trafficking, Youth and ...",[United States],2020,[Davy Rothbart],<NA>,2019
7,https://moviesthatmatter.nl/en/film/3-1-2-minu...,"3 1/2 Minutes, Ten Bullets",Electrifying documentary thriller about Afro-A...,"[Justice and law, Racism and/or discrimination...",[United States],2016,[Marc Silver],"[Orlando Bagwell, Su Patel]",2015
8,https://moviesthatmatter.nl/en/film/33-days,33 Days,"In the 2006 war between Israel and Lebanon, ar...","[Crimes against humanity and war crimes, Migra...",[Lebanon],2009,[Mai Masri],<NA>,2007
9,https://moviesthatmatter.nl/en/film/5-broken-c...,5 Broken Cameras,The major winner at IDFA 2011. Personal testim...,"[Activism and human rights defenders, House & ...","[Israel, Palestine]",2012,"[Emad Burnat, Guy Davidi]",<NA>,2011


They're numbers now! Okay, moving on...

## Scraping Country of Production

In [61]:
countries_of_production = []

for url in df_movies["movie_url"]:
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    countries = pd.NA

    credits = soup.find("dl", class_="data-list--details")
    if credits:
        items = credits.find_all("dt")
        values = credits.find_all("dd")

        for item, value in zip(items, values):
            if item.get_text(strip=True).lower() == "country of production":
                found = [s.strip() for s in value.stripped_strings if s.strip()]
                countries = found if found else pd.NA
                break

    countries_of_production.append(countries)

df_movies["country_of_production"] = countries_of_production
df_movies.head(10)


,movie_url,title,description,themes,relevant_countries,mtmf_year,director,producer,years,country_of_production
0,https://moviesthatmatter.nl/en/film/chicagogir...,#chicagoGirl – The Social Network Takes on a D...,The 19-year-old Ala’a Basatneh helps coordinat...,"[Activism and human rights defenders, Democrac...","[Syria, United States]",2014,[Joe Piscatella],"[Joe Piscatella, Mike Rinehart]",2013,"[United States, Syria]"
1,https://moviesthatmatter.nl/en/film/10-conditi...,The 10 Conditions of Love,Rebiya Kadeer once embodied China’s economic s...,"[Activism and human rights defenders, Culture ...","[China, Taiwan, Tibet, Turkey]",2010,[Jeff Daniels],[John Lewis],2009,"[United States, Australia]"
2,https://moviesthatmatter.nl/en/film/107-mothers,107 Mothers,"In a Ukrainian women’s prison, inmate Lesya ha...","[Prison and detention, Women's rights]",[Ukraine],2022,[Péter Kerekes],[Ivan Ostrochovský],2021,"[Slovakia, Czech Republic, Ukraine]"
3,https://moviesthatmatter.nl/en/film/12-years-a...,12 Years A Slave,Touching and thrilling movie based on the memo...,"[Democracy and politics, Racism and/or discrim...",[United States],2014,[Steve McQueen],"[Dede Gardner, Anthony Katagas]",2013,"[United States, United Kingdom]"
4,https://moviesthatmatter.nl/en/film/143-sahara...,143 Sahara Street,"In her small shop, in the vast Algerian Sahara...",[Women's rights],[Algeria],2020,[Hassen Ferhani],<NA>,2019,"[Algeria, Qatar, France]"
5,https://moviesthatmatter.nl/en/film/15-minutes...,15 minutes – The Dvor Massacre,"In the summer of 1995 Jørgen Kold, Danish offi...",[War and armed conflict],"[Bosnia and Herzegovina, Croatia, Denmark]",2016,"[Georg Larsen, Kasper Vedsmand]",[Anne Köhncke],2015,"[Croatia, Denmark]"
6,https://moviesthatmatter.nl/en/film/17-blocks,17 Blocks,Two decades in the life of the Sanford-Durant ...,"[Crime, Drugs and drug trafficking, Youth and ...",[United States],2020,[Davy Rothbart],<NA>,2019,[United States]
7,https://moviesthatmatter.nl/en/film/3-1-2-minu...,"3 1/2 Minutes, Ten Bullets",Electrifying documentary thriller about Afro-A...,"[Justice and law, Racism and/or discrimination...",[United States],2016,[Marc Silver],"[Orlando Bagwell, Su Patel]",2015,[United States]
8,https://moviesthatmatter.nl/en/film/33-days,33 Days,"In the 2006 war between Israel and Lebanon, ar...","[Crimes against humanity and war crimes, Migra...",[Lebanon],2009,[Mai Masri],<NA>,2007,[Lebanon]
9,https://moviesthatmatter.nl/en/film/5-broken-c...,5 Broken Cameras,The major winner at IDFA 2011. Personal testim...,"[Activism and human rights defenders, House & ...","[Israel, Palestine]",2012,"[Emad Burnat, Guy Davidi]",<NA>,2011,"[France, Israel, Palestine, Netherlands, The]"


## Scraping Duration 

In [64]:
durations = []

for url in df_movies["movie_url"]:
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    duration = pd.NA

    credits = soup.find("dl", class_="data-list--details")
    if credits:
        items = credits.find_all("dt")
        values = credits.find_all("dd")

        for item, value in zip(items, values):
            if item.get_text(strip=True).lower() == "duration":
                txt = value.get_text(strip=True)
                duration = txt if txt else pd.NA
                break

    durations.append(duration)

df_movies["duration"] = durations
df_movies.head(10)

,movie_url,title,description,themes,relevant_countries,mtmf_year,director,producer,years,country_of_production,duration
0,https://moviesthatmatter.nl/en/film/chicagogir...,#chicagoGirl – The Social Network Takes on a D...,The 19-year-old Ala’a Basatneh helps coordinat...,"[Activism and human rights defenders, Democrac...","[Syria, United States]",2014,[Joe Piscatella],"[Joe Piscatella, Mike Rinehart]",2013,"[United States, Syria]",74 minutes
1,https://moviesthatmatter.nl/en/film/10-conditi...,The 10 Conditions of Love,Rebiya Kadeer once embodied China’s economic s...,"[Activism and human rights defenders, Culture ...","[China, Taiwan, Tibet, Turkey]",2010,[Jeff Daniels],[John Lewis],2009,"[United States, Australia]",53 minutes
2,https://moviesthatmatter.nl/en/film/107-mothers,107 Mothers,"In a Ukrainian women’s prison, inmate Lesya ha...","[Prison and detention, Women's rights]",[Ukraine],2022,[Péter Kerekes],[Ivan Ostrochovský],2021,"[Slovakia, Czech Republic, Ukraine]",93 minutes
3,https://moviesthatmatter.nl/en/film/12-years-a...,12 Years A Slave,Touching and thrilling movie based on the memo...,"[Democracy and politics, Racism and/or discrim...",[United States],2014,[Steve McQueen],"[Dede Gardner, Anthony Katagas]",2013,"[United States, United Kingdom]",134 minutes
4,https://moviesthatmatter.nl/en/film/143-sahara...,143 Sahara Street,"In her small shop, in the vast Algerian Sahara...",[Women's rights],[Algeria],2020,[Hassen Ferhani],<NA>,2019,"[Algeria, Qatar, France]",104 minutes
5,https://moviesthatmatter.nl/en/film/15-minutes...,15 minutes – The Dvor Massacre,"In the summer of 1995 Jørgen Kold, Danish offi...",[War and armed conflict],"[Bosnia and Herzegovina, Croatia, Denmark]",2016,"[Georg Larsen, Kasper Vedsmand]",[Anne Köhncke],2015,"[Croatia, Denmark]",59 minutes
6,https://moviesthatmatter.nl/en/film/17-blocks,17 Blocks,Two decades in the life of the Sanford-Durant ...,"[Crime, Drugs and drug trafficking, Youth and ...",[United States],2020,[Davy Rothbart],<NA>,2019,[United States],96 minutes
7,https://moviesthatmatter.nl/en/film/3-1-2-minu...,"3 1/2 Minutes, Ten Bullets",Electrifying documentary thriller about Afro-A...,"[Justice and law, Racism and/or discrimination...",[United States],2016,[Marc Silver],"[Orlando Bagwell, Su Patel]",2015,[United States],98 minutes
8,https://moviesthatmatter.nl/en/film/33-days,33 Days,"In the 2006 war between Israel and Lebanon, ar...","[Crimes against humanity and war crimes, Migra...",[Lebanon],2009,[Mai Masri],<NA>,2007,[Lebanon],70 minutes
9,https://moviesthatmatter.nl/en/film/5-broken-c...,5 Broken Cameras,The major winner at IDFA 2011. Personal testim...,"[Activism and human rights defenders, House & ...","[Israel, Palestine]",2012,"[Emad Burnat, Guy Davidi]",<NA>,2011,"[France, Israel, Palestine, Netherlands, The]",90 minutes


Notice that the output of this code is in the format of 'x minutes', meaning it is again a string! But we want numbers...

## Exercise
In this tutorial, you may have noticed two ways of converting a string into numerical: one using __isdigit()__ and one using __regular expressions and astype()__. You are asked to use the regular expressions and astype() and get our duration column from e.g.: 90 minutes (string) to 90 (numerical).

In [ ]:
#type your code here...






*__Solution:__*

In [65]:
df_movies["duration"] = (
    df_movies["duration"]
    .str.extract(r"(\d+)")
    .astype("Int64"))
df_movies.head(10)

,movie_url,title,description,themes,relevant_countries,mtmf_year,director,producer,years,country_of_production,duration
0,https://moviesthatmatter.nl/en/film/chicagogir...,#chicagoGirl – The Social Network Takes on a D...,The 19-year-old Ala’a Basatneh helps coordinat...,"[Activism and human rights defenders, Democrac...","[Syria, United States]",2014,[Joe Piscatella],"[Joe Piscatella, Mike Rinehart]",2013,"[United States, Syria]",74
1,https://moviesthatmatter.nl/en/film/10-conditi...,The 10 Conditions of Love,Rebiya Kadeer once embodied China’s economic s...,"[Activism and human rights defenders, Culture ...","[China, Taiwan, Tibet, Turkey]",2010,[Jeff Daniels],[John Lewis],2009,"[United States, Australia]",53
2,https://moviesthatmatter.nl/en/film/107-mothers,107 Mothers,"In a Ukrainian women’s prison, inmate Lesya ha...","[Prison and detention, Women's rights]",[Ukraine],2022,[Péter Kerekes],[Ivan Ostrochovský],2021,"[Slovakia, Czech Republic, Ukraine]",93
3,https://moviesthatmatter.nl/en/film/12-years-a...,12 Years A Slave,Touching and thrilling movie based on the memo...,"[Democracy and politics, Racism and/or discrim...",[United States],2014,[Steve McQueen],"[Dede Gardner, Anthony Katagas]",2013,"[United States, United Kingdom]",134
4,https://moviesthatmatter.nl/en/film/143-sahara...,143 Sahara Street,"In her small shop, in the vast Algerian Sahara...",[Women's rights],[Algeria],2020,[Hassen Ferhani],<NA>,2019,"[Algeria, Qatar, France]",104
5,https://moviesthatmatter.nl/en/film/15-minutes...,15 minutes – The Dvor Massacre,"In the summer of 1995 Jørgen Kold, Danish offi...",[War and armed conflict],"[Bosnia and Herzegovina, Croatia, Denmark]",2016,"[Georg Larsen, Kasper Vedsmand]",[Anne Köhncke],2015,"[Croatia, Denmark]",59
6,https://moviesthatmatter.nl/en/film/17-blocks,17 Blocks,Two decades in the life of the Sanford-Durant ...,"[Crime, Drugs and drug trafficking, Youth and ...",[United States],2020,[Davy Rothbart],<NA>,2019,[United States],96
7,https://moviesthatmatter.nl/en/film/3-1-2-minu...,"3 1/2 Minutes, Ten Bullets",Electrifying documentary thriller about Afro-A...,"[Justice and law, Racism and/or discrimination...",[United States],2016,[Marc Silver],"[Orlando Bagwell, Su Patel]",2015,[United States],98
8,https://moviesthatmatter.nl/en/film/33-days,33 Days,"In the 2006 war between Israel and Lebanon, ar...","[Crimes against humanity and war crimes, Migra...",[Lebanon],2009,[Mai Masri],<NA>,2007,[Lebanon],70
9,https://moviesthatmatter.nl/en/film/5-broken-c...,5 Broken Cameras,The major winner at IDFA 2011. Personal testim...,"[Activism and human rights defenders, House & ...","[Israel, Palestine]",2012,"[Emad Burnat, Guy Davidi]",<NA>,2011,"[France, Israel, Palestine, Netherlands, The]",90


That's more like it!

## Scraping Spoken Languages
Now we are going to scrape the spoken languages of each movie:

In [66]:
spoken_languages = []

for url in df_movies["movie_url"]:
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    spoken_language = pd.NA

    credits = soup.find("dl", class_="data-list--details")
    if credits:
        items = credits.find_all("dt")
        values = credits.find_all("dd")

        for item, value in zip(items, values):
            if item.get_text(strip=True).lower() == "spoken language":
                found = [s.strip() for s in value.stripped_strings if s.strip()]
                spoken_language = found if found else pd.NA
                break

    spoken_languages.append(spoken_language)

df_movies["spoken_languages"] = spoken_languages
df_movies.head(10)


,movie_url,title,description,themes,relevant_countries,mtmf_year,director,producer,years,country_of_production,duration,spoken_languages
0,https://moviesthatmatter.nl/en/film/chicagogir...,#chicagoGirl – The Social Network Takes on a D...,The 19-year-old Ala’a Basatneh helps coordinat...,"[Activism and human rights defenders, Democrac...","[Syria, United States]",2014,[Joe Piscatella],"[Joe Piscatella, Mike Rinehart]",2013,"[United States, Syria]",74,"[Arabic, English]"
1,https://moviesthatmatter.nl/en/film/10-conditi...,The 10 Conditions of Love,Rebiya Kadeer once embodied China’s economic s...,"[Activism and human rights defenders, Culture ...","[China, Taiwan, Tibet, Turkey]",2010,[Jeff Daniels],[John Lewis],2009,"[United States, Australia]",53,[English]
2,https://moviesthatmatter.nl/en/film/107-mothers,107 Mothers,"In a Ukrainian women’s prison, inmate Lesya ha...","[Prison and detention, Women's rights]",[Ukraine],2022,[Péter Kerekes],[Ivan Ostrochovský],2021,"[Slovakia, Czech Republic, Ukraine]",93,"[Russian, Ukrainian]"
3,https://moviesthatmatter.nl/en/film/12-years-a...,12 Years A Slave,Touching and thrilling movie based on the memo...,"[Democracy and politics, Racism and/or discrim...",[United States],2014,[Steve McQueen],"[Dede Gardner, Anthony Katagas]",2013,"[United States, United Kingdom]",134,[English]
4,https://moviesthatmatter.nl/en/film/143-sahara...,143 Sahara Street,"In her small shop, in the vast Algerian Sahara...",[Women's rights],[Algeria],2020,[Hassen Ferhani],<NA>,2019,"[Algeria, Qatar, France]",104,[Arabic]
5,https://moviesthatmatter.nl/en/film/15-minutes...,15 minutes – The Dvor Massacre,"In the summer of 1995 Jørgen Kold, Danish offi...",[War and armed conflict],"[Bosnia and Herzegovina, Croatia, Denmark]",2016,"[Georg Larsen, Kasper Vedsmand]",[Anne Köhncke],2015,"[Croatia, Denmark]",59,"[Croatian, Danish, English, Serbian]"
6,https://moviesthatmatter.nl/en/film/17-blocks,17 Blocks,Two decades in the life of the Sanford-Durant ...,"[Crime, Drugs and drug trafficking, Youth and ...",[United States],2020,[Davy Rothbart],<NA>,2019,[United States],96,[English]
7,https://moviesthatmatter.nl/en/film/3-1-2-minu...,"3 1/2 Minutes, Ten Bullets",Electrifying documentary thriller about Afro-A...,"[Justice and law, Racism and/or discrimination...",[United States],2016,[Marc Silver],"[Orlando Bagwell, Su Patel]",2015,[United States],98,[English]
8,https://moviesthatmatter.nl/en/film/33-days,33 Days,"In the 2006 war between Israel and Lebanon, ar...","[Crimes against humanity and war crimes, Migra...",[Lebanon],2009,[Mai Masri],<NA>,2007,[Lebanon],70,[Arabic]
9,https://moviesthatmatter.nl/en/film/5-broken-c...,5 Broken Cameras,The major winner at IDFA 2011. Personal testim...,"[Activism and human rights defenders, House & ...","[Israel, Palestine]",2012,"[Emad Burnat, Guy Davidi]",<NA>,2011,"[France, Israel, Palestine, Netherlands, The]",90,"[Arabic, Hebrew]"


## Scraping Subtitles
Here, we are also going to preserve the subtitles available at the festival for each movie.

In [67]:
subtitles = []

for url in df_movies["movie_url"]:
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    subtitle = pd.NA

    credits = soup.find("dl", class_="data-list--details")
    if credits:
        items = credits.find_all("dt")
        values = credits.find_all("dd")

        for item, value in zip(items, values):
            if item.get_text(strip=True).lower() == "subtitles":
                found = [s.strip() for s in value.stripped_strings if s.strip()]
                subtitle = found if found else pd.NA
                break

    subtitles.append(subtitle)

df_movies["subtitles"] = subtitles
df_movies.head(10)


,movie_url,title,description,themes,relevant_countries,mtmf_year,director,producer,years,country_of_production,duration,spoken_languages,subtitles
0,https://moviesthatmatter.nl/en/film/chicagogir...,#chicagoGirl – The Social Network Takes on a D...,The 19-year-old Ala’a Basatneh helps coordinat...,"[Activism and human rights defenders, Democrac...","[Syria, United States]",2014,[Joe Piscatella],"[Joe Piscatella, Mike Rinehart]",2013,"[United States, Syria]",74,"[Arabic, English]",[EN]
1,https://moviesthatmatter.nl/en/film/10-conditi...,The 10 Conditions of Love,Rebiya Kadeer once embodied China’s economic s...,"[Activism and human rights defenders, Culture ...","[China, Taiwan, Tibet, Turkey]",2010,[Jeff Daniels],[John Lewis],2009,"[United States, Australia]",53,[English],[EN]
2,https://moviesthatmatter.nl/en/film/107-mothers,107 Mothers,"In a Ukrainian women’s prison, inmate Lesya ha...","[Prison and detention, Women's rights]",[Ukraine],2022,[Péter Kerekes],[Ivan Ostrochovský],2021,"[Slovakia, Czech Republic, Ukraine]",93,"[Russian, Ukrainian]",[NL]
3,https://moviesthatmatter.nl/en/film/12-years-a...,12 Years A Slave,Touching and thrilling movie based on the memo...,"[Democracy and politics, Racism and/or discrim...",[United States],2014,[Steve McQueen],"[Dede Gardner, Anthony Katagas]",2013,"[United States, United Kingdom]",134,[English],[NL]
4,https://moviesthatmatter.nl/en/film/143-sahara...,143 Sahara Street,"In her small shop, in the vast Algerian Sahara...",[Women's rights],[Algeria],2020,[Hassen Ferhani],<NA>,2019,"[Algeria, Qatar, France]",104,[Arabic],[EN]
5,https://moviesthatmatter.nl/en/film/15-minutes...,15 minutes – The Dvor Massacre,"In the summer of 1995 Jørgen Kold, Danish offi...",[War and armed conflict],"[Bosnia and Herzegovina, Croatia, Denmark]",2016,"[Georg Larsen, Kasper Vedsmand]",[Anne Köhncke],2015,"[Croatia, Denmark]",59,"[Croatian, Danish, English, Serbian]",[EN]
6,https://moviesthatmatter.nl/en/film/17-blocks,17 Blocks,Two decades in the life of the Sanford-Durant ...,"[Crime, Drugs and drug trafficking, Youth and ...",[United States],2020,[Davy Rothbart],<NA>,2019,[United States],96,[English],<NA>
7,https://moviesthatmatter.nl/en/film/3-1-2-minu...,"3 1/2 Minutes, Ten Bullets",Electrifying documentary thriller about Afro-A...,"[Justice and law, Racism and/or discrimination...",[United States],2016,[Marc Silver],"[Orlando Bagwell, Su Patel]",2015,[United States],98,[English],<NA>
8,https://moviesthatmatter.nl/en/film/33-days,33 Days,"In the 2006 war between Israel and Lebanon, ar...","[Crimes against humanity and war crimes, Migra...",[Lebanon],2009,[Mai Masri],<NA>,2007,[Lebanon],70,[Arabic],[EN]
9,https://moviesthatmatter.nl/en/film/5-broken-c...,5 Broken Cameras,The major winner at IDFA 2011. Personal testim...,"[Activism and human rights defenders, House & ...","[Israel, Palestine]",2012,"[Emad Burnat, Guy Davidi]",<NA>,2011,"[France, Israel, Palestine, Netherlands, The]",90,"[Arabic, Hebrew]",[EN]


## Scraping Production Company

In [68]:
production_companies = []

for url in df_movies["movie_url"]:
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    production_company = pd.NA

    credits = soup.find("dl", class_="data-list--details")
    if credits:
        items = credits.find_all("dt")
        values = credits.find_all("dd")

        for item, value in zip(items, values):
            if item.get_text(strip=True).lower() == "production company":
                found = [s.strip() for s in value.stripped_strings if s.strip()]
                production_company = found if found else pd.NA
                break

    production_companies.append(production_company)

df_movies["production_companies"] = production_companies
df_movies.head(10)

,movie_url,title,description,themes,relevant_countries,mtmf_year,director,producer,years,country_of_production,duration,spoken_languages,subtitles,production_companies
0,https://moviesthatmatter.nl/en/film/chicagogir...,#chicagoGirl – The Social Network Takes on a D...,The 19-year-old Ala’a Basatneh helps coordinat...,"[Activism and human rights defenders, Democrac...","[Syria, United States]",2014,[Joe Piscatella],"[Joe Piscatella, Mike Rinehart]",2013,"[United States, Syria]",74,"[Arabic, English]",[EN],[Revolutio LLC.]
1,https://moviesthatmatter.nl/en/film/10-conditi...,The 10 Conditions of Love,Rebiya Kadeer once embodied China’s economic s...,"[Activism and human rights defenders, Culture ...","[China, Taiwan, Tibet, Turkey]",2010,[Jeff Daniels],[John Lewis],2009,"[United States, Australia]",53,[English],[EN],[Arcimedia]
2,https://moviesthatmatter.nl/en/film/107-mothers,107 Mothers,"In a Ukrainian women’s prison, inmate Lesya ha...","[Prison and detention, Women's rights]",[Ukraine],2022,[Péter Kerekes],[Ivan Ostrochovský],2021,"[Slovakia, Czech Republic, Ukraine]",93,"[Russian, Ukrainian]",[NL],[Punkchart Films]
3,https://moviesthatmatter.nl/en/film/12-years-a...,12 Years A Slave,Touching and thrilling movie based on the memo...,"[Democracy and politics, Racism and/or discrim...",[United States],2014,[Steve McQueen],"[Dede Gardner, Anthony Katagas]",2013,"[United States, United Kingdom]",134,[English],[NL],[New Regency Pictures]
4,https://moviesthatmatter.nl/en/film/143-sahara...,143 Sahara Street,"In her small shop, in the vast Algerian Sahara...",[Women's rights],[Algeria],2020,[Hassen Ferhani],<NA>,2019,"[Algeria, Qatar, France]",104,[Arabic],[EN],<NA>
5,https://moviesthatmatter.nl/en/film/15-minutes...,15 minutes – The Dvor Massacre,"In the summer of 1995 Jørgen Kold, Danish offi...",[War and armed conflict],"[Bosnia and Herzegovina, Croatia, Denmark]",2016,"[Georg Larsen, Kasper Vedsmand]",[Anne Köhncke],2015,"[Croatia, Denmark]",59,"[Croatian, Danish, English, Serbian]",[EN],[Final Cut for Real ApS]
6,https://moviesthatmatter.nl/en/film/17-blocks,17 Blocks,Two decades in the life of the Sanford-Durant ...,"[Crime, Drugs and drug trafficking, Youth and ...",[United States],2020,[Davy Rothbart],<NA>,2019,[United States],96,[English],<NA>,[Beachside Films]
7,https://moviesthatmatter.nl/en/film/3-1-2-minu...,"3 1/2 Minutes, Ten Bullets",Electrifying documentary thriller about Afro-A...,"[Justice and law, Racism and/or discrimination...",[United States],2016,[Marc Silver],"[Orlando Bagwell, Su Patel]",2015,[United States],98,[English],<NA>,[Motto Pictures]
8,https://moviesthatmatter.nl/en/film/33-days,33 Days,"In the 2006 war between Israel and Lebanon, ar...","[Crimes against humanity and war crimes, Migra...",[Lebanon],2009,[Mai Masri],<NA>,2007,[Lebanon],70,[Arabic],[EN],[Nour Productions]
9,https://moviesthatmatter.nl/en/film/5-broken-c...,5 Broken Cameras,The major winner at IDFA 2011. Personal testim...,"[Activism and human rights defenders, House & ...","[Israel, Palestine]",2012,"[Emad Burnat, Guy Davidi]",<NA>,2011,"[France, Israel, Palestine, Netherlands, The]",90,"[Arabic, Hebrew]",[EN],[Guy DVD Films]


## Scraping World Sales

In [69]:
world_sales = []

for url in df_movies["movie_url"]:
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    world_sale = pd.NA

    credits = soup.find("dl", class_="data-list--details")
    if credits:
        items = credits.find_all("dt")
        values = credits.find_all("dd")

        for item, value in zip(items, values):
            if item.get_text(strip=True).lower() == "world sales":
                found = [s.strip() for s in value.stripped_strings if s.strip()]
                world_sale = found if found else pd.NA
                break

    world_sales.append(world_sale)

df_movies["world_sales"] = world_sales
df_movies.head(10)


,movie_url,title,description,themes,relevant_countries,mtmf_year,director,producer,years,country_of_production,duration,spoken_languages,subtitles,production_companies,world_sales
0,https://moviesthatmatter.nl/en/film/chicagogir...,#chicagoGirl – The Social Network Takes on a D...,The 19-year-old Ala’a Basatneh helps coordinat...,"[Activism and human rights defenders, Democrac...","[Syria, United States]",2014,[Joe Piscatella],"[Joe Piscatella, Mike Rinehart]",2013,"[United States, Syria]",74,"[Arabic, English]",[EN],[Revolutio LLC.],[Ro*co Films International]
1,https://moviesthatmatter.nl/en/film/10-conditi...,The 10 Conditions of Love,Rebiya Kadeer once embodied China’s economic s...,"[Activism and human rights defenders, Culture ...","[China, Taiwan, Tibet, Turkey]",2010,[Jeff Daniels],[John Lewis],2009,"[United States, Australia]",53,[English],[EN],[Arcimedia],[TVF International]
2,https://moviesthatmatter.nl/en/film/107-mothers,107 Mothers,"In a Ukrainian women’s prison, inmate Lesya ha...","[Prison and detention, Women's rights]",[Ukraine],2022,[Péter Kerekes],[Ivan Ostrochovský],2021,"[Slovakia, Czech Republic, Ukraine]",93,"[Russian, Ukrainian]",[NL],[Punkchart Films],[Films Boutique]
3,https://moviesthatmatter.nl/en/film/12-years-a...,12 Years A Slave,Touching and thrilling movie based on the memo...,"[Democracy and politics, Racism and/or discrim...",[United States],2014,[Steve McQueen],"[Dede Gardner, Anthony Katagas]",2013,"[United States, United Kingdom]",134,[English],[NL],[New Regency Pictures],[Lionsgate]
4,https://moviesthatmatter.nl/en/film/143-sahara...,143 Sahara Street,"In her small shop, in the vast Algerian Sahara...",[Women's rights],[Algeria],2020,[Hassen Ferhani],<NA>,2019,"[Algeria, Qatar, France]",104,[Arabic],[EN],<NA>,[Pascale Ramonda]
5,https://moviesthatmatter.nl/en/film/15-minutes...,15 minutes – The Dvor Massacre,"In the summer of 1995 Jørgen Kold, Danish offi...",[War and armed conflict],"[Bosnia and Herzegovina, Croatia, Denmark]",2016,"[Georg Larsen, Kasper Vedsmand]",[Anne Köhncke],2015,"[Croatia, Denmark]",59,"[Croatian, Danish, English, Serbian]",[EN],[Final Cut for Real ApS],[DR Sales]
6,https://moviesthatmatter.nl/en/film/17-blocks,17 Blocks,Two decades in the life of the Sanford-Durant ...,"[Crime, Drugs and drug trafficking, Youth and ...",[United States],2020,[Davy Rothbart],<NA>,2019,[United States],96,[English],<NA>,[Beachside Films],[Cinephil]
7,https://moviesthatmatter.nl/en/film/3-1-2-minu...,"3 1/2 Minutes, Ten Bullets",Electrifying documentary thriller about Afro-A...,"[Justice and law, Racism and/or discrimination...",[United States],2016,[Marc Silver],"[Orlando Bagwell, Su Patel]",2015,[United States],98,[English],<NA>,[Motto Pictures],[Dogwoof]
8,https://moviesthatmatter.nl/en/film/33-days,33 Days,"In the 2006 war between Israel and Lebanon, ar...","[Crimes against humanity and war crimes, Migra...",[Lebanon],2009,[Mai Masri],<NA>,2007,[Lebanon],70,[Arabic],[EN],[Nour Productions],[Arab Film Distribution / Typecast Releasing]
9,https://moviesthatmatter.nl/en/film/5-broken-c...,5 Broken Cameras,The major winner at IDFA 2011. Personal testim...,"[Activism and human rights defenders, House & ...","[Israel, Palestine]",2012,"[Emad Burnat, Guy Davidi]",<NA>,2011,"[France, Israel, Palestine, Netherlands, The]",90,"[Arabic, Hebrew]",[EN],[Guy DVD Films],[Cat & Docs]


## Final Step: Saving the CSV file :)

In [70]:
df_movies.to_csv("df_mtmf.csv", index=False)

The end.